# Vector RAG Pipeline — FinanceBench

**Paper:** Kim et al. (2025), _GAR: Generative Answer Refinement for Financial QA_, arXiv 2503.15191  
**Thesis:** Vector RAG vs. Vectorless RAG vs. Long-Context LLMs on FinanceBench

---

## What we're building

This notebook builds the Vector RAG pipeline **one stage at a time**, with explanations before each stage so you understand the _why_ before the _how_.

The full pipeline has **7 stages**:

```
Question
  │
  ▼
① Load data          ── pick a question, find its PDF
  │
  ▼
② PDF → Markdown → Chunks  ── convert the raw PDF into 512-token passages,
  │                              each tagged with its page number
  ▼
③ Index              ── embed every chunk with Stella 1.5B (dense vectors)
  │                     + build a BM25 index (sparse, keyword-based)
  ▼
④ Query expansion    ── ask an LLM to rewrite/expand the question so it
  │                     better matches how the answer is phrased in the doc
  ▼
⑤ Hybrid retrieval   ── combine dense + sparse scores:
  │                     hybrid = 0.85 × dense + 0.15 × sparse
  │                     return top-20 chunks
  ▼
⑥ Rerank             ── Voyage rerank-2 API re-scores all 20 chunk-question
  │                     pairs and keeps only the top-10
  ▼
⑦ Generate           ── selection agent filters the 10 chunks to the truly
                         relevant ones, then Gemini / DeepSeek produces the answer
```

We'll build one stage per notebook section. By the end, you'll be able to feed in any FinanceBench question and get an answer, with full cost and latency logging.

> **Running locally vs. on Colab**  
> Stages ①–② and ④–⑦ are fine locally (they're API calls or small computations).  
> Stage ③ (embedding chunks with a 1.5B-parameter model) is slow on CPU.  
> For now we test on **one document** — manageable locally. Full 84-doc indexing goes to Colab.

---
## Stage 0 — Setup

Install dependencies and load API keys from `.env`.

The `%pip install` cells work both locally (inside the uv venv) and on Colab without changes.

In [ ]:
# Core deps — needed for all stages.
%pip install -q pandas python-dotenv google-genai openai voyageai anthropic
# Stage 2 deps (PDF parsing, tokenizer, chunking)
%pip install -q pymupdf4llm transformers nltk
# Stage 3 deps (embedding model, sparse retrieval) — large download the first time
%pip install -q sentence-transformers rank_bm25

: 

In [1]:
import os, sys, json, time, textwrap
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# ── Repo root detection ────────────────────────────────────────────────────
# Works regardless of where you launched Jupyter from (repo root, notebook
# folder, or Colab). Walks up until it finds data/financebench_open_source.jsonl.
_sentinel = "data/financebench_open_source.jsonl"
REPO_ROOT = None
for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_p / _sentinel).exists():
        REPO_ROOT = _p
        break
if REPO_ROOT is None:
    # On Colab: set this manually after uploading your repo
    REPO_ROOT = Path("/content/financebench-rag-thesis")
    print(f"⚠ Could not auto-detect repo root. Using {REPO_ROOT}")
    print("  If wrong, set REPO_ROOT manually below and re-run.")

DATA_DIR = REPO_ROOT / "data"
PDF_DIR  = REPO_ROOT / "pdfs"
sys.path.insert(0, str(REPO_ROOT))   # so we can import evaluation/cost_tracker
print(f"REPO_ROOT: {REPO_ROOT}")

# ── API keys ───────────────────────────────────────────────────────────────
load_dotenv(REPO_ROOT / ".env")
GOOGLE_API_KEY    = os.getenv("GOOGLE_API_KEY", "")
DEEPSEEK_API_KEY  = os.getenv("DEEPSEEK_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
VOYAGE_API_KEY    = os.getenv("VOYAGE_API_KEY", "")

print("\nAPI keys:")
for name, val in [("GOOGLE_API_KEY", GOOGLE_API_KEY), ("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
                  ("ANTHROPIC_API_KEY", ANTHROPIC_API_KEY), ("VOYAGE_API_KEY", VOYAGE_API_KEY)]:
    print(f"  {name:<20}: {'✓' if val else '✗  (not set)'}")

REPO_ROOT: /Users/shaliqshukoor/dev/thesis-project

API keys:
  GOOGLE_API_KEY      : ✗  (not set)
  DEEPSEEK_API_KEY    : ✗  (not set)
  ANTHROPIC_API_KEY   : ✗  (not set)
  VOYAGE_API_KEY      : ✗  (not set)


---
## Stage 1 — Load FinanceBench data and pick a test question

### What is FinanceBench?

FinanceBench (Islam et al., 2023) is a benchmark of **150 questions** about real SEC filings — 10-K annual reports, 10-Q quarterly reports, and earnings releases. Every question has:

- a **gold answer** (human-annotated)
- **evidence**: the exact page(s) in the source PDF where the answer can be found (`evidence_page_num`)
- a **reasoning type** label (e.g. `"arithmetic"`, `"multi-hop"`, `"lookup"`)

That `evidence_page_num` field is gold for us: we'll use it to evaluate *retrieval quality* — did our pipeline retrieve the right page? — independently of *answer quality* — did the generator produce the right answer?

### The two files

| File | What it contains |
|------|------------------|
| `financebench_open_source.jsonl` | 150 questions — question text, gold answer, evidence with page numbers |
| `financebench_document_information.jsonl` | Metadata for each source document — company, sector, filing type, year |

In [2]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

print(f"Total questions : {len(df)}")
print(f"Unique documents: {df.doc_name.nunique()}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Total questions : 150
Unique documents: 84
Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,NaN,What is the FY2018 capital expenditure amount ...,$1577.00,The metric capital expenditures was directly e...,OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,NaN,Assume that you are a public equities analyst....,$8.70,"The metric ppne, net was directly extracted fr...",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=...,OPEN_SOURCE,[{'evidence_text': '3M Company and Subsidiarie...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...


In [3]:
# Change TEST_IDX (0–149) to try a different question.
TEST_IDX = 0
row = df.iloc[TEST_IDX]

print("=" * 70)
print(f"Question ID   : {row.financebench_id}")
print(f"Company       : {row.company}")
print(f"Document      : {row.doc_name}  ({row.doc_type.upper()}, {row.doc_period})")
print(f"Reasoning type: {row.question_reasoning}")
print("=" * 70)
print(f"\nQuestion:\n  {row.question}")
print(f"\nGold answer:\n  {row.answer}")

evidence = row.evidence  # list of dicts; evidence_page_num is ZERO-indexed
print(f"\nGold evidence page(s) [0-indexed]: {[e['evidence_page_num'] for e in evidence]}")

Question ID   : financebench_id_03029
Company       : 3M
Document      : 3M_2018_10K  (10K, 2018)
Reasoning type: Information extraction

Question:
  What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

Gold answer:
  $1577.00

Gold evidence page(s) [0-indexed]: [59]


In [4]:
pdf_path = PDF_DIR / f"{row.doc_name}.pdf"

if pdf_path.exists():
    print(f"✓ PDF found: {pdf_path.name}  ({pdf_path.stat().st_size / 1_048_576:.1f} MB)")
else:
    print(f"✗ PDF not found at {pdf_path}")

# These variables flow into every later stage
TEST_DOC_NAME      = row.doc_name
TEST_QUESTION      = row.question
TEST_ANSWER        = row.answer
TEST_EVIDENCE_PAGES = [e['evidence_page_num'] for e in row.evidence]
TEST_PDF_PATH      = pdf_path

✓ PDF found: 3M_2018_10K.pdf  (1.2 MB)


---
## Stage 2 — PDF → Markdown → 512-token chunks

### Why do we chunk at all?

An embedding model converts text into a single fixed-size vector (a list of numbers that encodes the meaning of the text). If you embed an **entire 200-page 10-K** as one vector, you get a vague average of everything in the document — it'll match almost any financial question, which makes it useless for retrieval.

Instead we split the document into small passages and embed each one. When a question comes in, we compare it against all passage embeddings and find the ones with the closest meaning. Smaller passages = more precise matches.

### Why 512 tokens?

512 tokens ≈ 380 words ≈ roughly half a typical 10-K page. It's a sweet spot:
- **Too small** (e.g. 64 tokens): a passage loses context, making it hard to understand what the number refers to
- **Too large** (e.g. 2048 tokens): the embedding becomes a blur again, retrieval gets imprecise
- **512** is also the standard max input length for many embedding models (Stella 1.5B can handle longer, but the paper uses 512)

### Why convert to Markdown first?

Raw PDF text extraction loses structure. A table becomes a jumbled sequence of numbers with no row/column headers. In a 10-K, that means a number like `1,234.5` is meaningless without knowing it's "Net revenue, Q3 2022".

Markdown conversion (via `pymupdf4llm`) preserves:
- **Table structure** as pipe-separated Markdown tables — headers stay attached to values
- **Headings** (e.g. `## NOTES TO FINANCIAL STATEMENTS`) — gives the model context about which section a chunk belongs to
- **Bold/italic emphasis** — often used for key financial figures

### Why chunk within page boundaries?

FinanceBench's gold evidence is expressed as a `evidence_page_num` — a specific page number. To evaluate retrieval quality ("did we retrieve the right page?"), every chunk must carry an unambiguous page number. If we let chunks span page boundaries, a chunk spanning pages 5–6 creates ambiguity: does retrieving it count as a hit for page 5 or page 6?

**Our design choice**: chunk *within* each page. This is a small deviation from the paper (which just says "512-token passages" without specifying page-boundary behaviour), but it makes the evaluation clean and unambiguous.

### Why overlap?

A sentence that sits at the boundary between two chunks shouldn't be arbitrarily cut. We carry over ~50 tokens from the end of each chunk into the beginning of the next one, so boundary sentences appear in both chunks. This avoids losing context at split points.

In [5]:
import re
import pymupdf4llm
import tiktoken

# ── Sentence splitter (no NLTK needed for English-only SEC filings) ────────
def sent_tokenize(text: str) -> list[str]:
    sentences = re.split(r'(?<!\d)\.[ \n]+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if s.strip()]

# ── Token counter ──────────────────────────────────────────────────────────
# We use tiktoken (cl100k, same vocab as GPT-4) instead of loading the full
# Stella tokenizer. Counts are within ~5% of Stella's own tokenizer for
# English text — more than accurate enough for 512-token chunking.
# Stella's tokenizer (and the full 1.5B model weights) loads in Stage 3
# when we actually need to embed chunks — not here.
_enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(_enc.encode(text))

# Sanity check
_test = "Net revenue was $1,234.5 million. Operating income increased 12.3%. Cash flows were strong."
print(f"✓ tiktoken ready — test sentence: {count_tokens(_test)} tokens")
print(f"✓ sentence splitter — {len(sent_tokenize(_test))} sentences")

✓ tiktoken ready — test sentence: 24 tokens
✓ sentence splitter — 3 sentences


In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────────
# We load only the tokenizer config (~5 MB), NOT the 1.5B model weights.
# This gives us accurate token counts using the same vocabulary the embedding
# model sees — so "512 tokens" means exactly 512 tokens to Stella.
STELLA_MODEL = "NovaSearch/stella_en_1.5B_v5"
print(f"Loading tokenizer for {STELLA_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(STELLA_MODEL, trust_remote_code=True)
print(f"✓ Tokenizer ready ({tokenizer.__class__.__name__})")

In [6]:
# ── Step 1: PDF → per-page Markdown ───────────────────────────────────────
# pymupdf4llm.to_markdown() with page_chunks=True returns a list of dicts,
# one per page.  Confirmed keys (from running on 3M_2018_10K.pdf):
#   "text"     : markdown string for that page
#   "metadata" : dict with "page_number" (1-indexed!), "page_count", etc.
#
# ⚠ Indexing trap: pymupdf4llm uses 1-indexed page_number (1, 2, 3…),
# but FinanceBench evidence_page_num is 0-indexed (0, 1, 2…).
# We subtract 1 when storing page_num so our chunks always match
# FinanceBench's convention: stored page_num == evidence_page_num.

print(f"Extracting markdown from {TEST_PDF_PATH.name} ...")
print("(This may take ~1 min locally if OCR is triggered — faster on Colab GPU)")
t0 = time.time()
md_pages = pymupdf4llm.to_markdown(str(TEST_PDF_PATH), page_chunks=True)
elapsed = time.time() - t0

print(f"✓ Done in {elapsed:.1f}s — {len(md_pages)} pages")
print(f"\nKeys in each page dict : {list(md_pages[0].keys())}")
print(f"Metadata keys          : {list(md_pages[0].get('metadata', {}).keys())}")
print(f"Example page_number    : {md_pages[0]['metadata']['page_number']}  ← 1-indexed")

Extracting markdown from 3M_2018_10K.pdf ...
(This may take ~1 min locally if OCR is triggered — faster on Colab GPU)
=== Document parser messages ===
Using Tesseract for OCR processing.

✓ Done in 26.2s — 160 pages

Keys in each page dict : ['metadata', 'toc_items', 'page_boxes', 'text']
Metadata keys          : ['format', 'title', 'author', 'subject', 'keywords', 'creator', 'producer', 'creationDate', 'modDate', 'trapped', 'encryption', 'file_path', 'page_count', 'page_number']
Example page_number    : 1  ← 1-indexed


In [9]:
print(md_pages)


[defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x113d40540>, {'metadata': {'format': 'PDF 1.4', 'title': '10-K - 02/07/2019 - 3M Company', 'author': '', 'subject': '', 'keywords': '', 'creator': 'wkhtmltopdf 0.12.5', 'producer': 'Qt 4.8.7', 'creationDate': "D:20230105161720-05'00'", 'modDate': '', 'trapped': '', 'encryption': None, 'file_path': '/Users/shaliqshukoor/dev/thesis-project/pdfs/3M_2018_10K.pdf', 'page_count': 160, 'page_number': 1}, 'toc_items': [], 'page_boxes': [{'index': 0, 'class': 'page-header', 'bbox': (81, 22, 83, 23), 'pos': (0, 6)}, {'index': 1, 'class': 'title', 'bbox': (165, 31, 437, 56), 'pos': (6, 63)}, {'index': 2, 'class': 'text', 'bbox': (263, 59, 338, 67), 'pos': (63, 92)}, {'index': 3, 'class': 'title', 'bbox': (266, 74, 335, 84), 'pos': (92, 110)}, {'index': 4, 'class': 'title', 'bbox': (186, 92, 416, 100), 'pos': (110, 178)}, {'index': 5, 'class': 'text', 'bbox': (234, 101, 367, 107), 'pos': (178, 216)}, {'index': 6, 'class': 'title', 'bbox'

In [10]:
import json

with open("parsed_output.json", "w", encoding="utf-8") as f:
    json.dump(md_pages, f, indent=2, default=str, ensure_ascii=False)

In [11]:
# Preview the gold evidence page.
# FinanceBench evidence_page_num=59 (0-indexed) → pymupdf4llm page_number=60 (1-indexed)
# We index into md_pages list using the 0-indexed value directly (list index = 0-indexed).
gold_page_0idx = TEST_EVIDENCE_PAGES[0]          # 0-indexed, matches FinanceBench
gold_md = md_pages[gold_page_0idx]               # list[59] = the 60th page

print(f"FinanceBench evidence_page_num : {gold_page_0idx}  (0-indexed)")
print(f"pymupdf4llm page_number        : {gold_md['metadata']['page_number']}  (1-indexed)")
print(f"\n--- First 1500 chars of the gold page's markdown ---")
print(gold_md["text"][:1500])

FinanceBench evidence_page_num : 59  (0-indexed)
pymupdf4llm page_number        : 60  (1-indexed)

--- First 1500 chars of the gold page's markdown ---
<u>Table of Contents</u> 

### **3M Company and Subsidiaries Consolidated Statement of Cash Flow s Years ended December 31** 

|**(Millions)**|**2018**|**2017**|**2016**|
|---|---|---|---|
|**Cash Flows from Operating Activities**||||
|<br>Net income including noncontrolling interest|**$**<br>**5,363**|$ 4,869|$ 5,058|
|Adjustments to reconcile net income including noncontrolling interest to net cash||||
|provided by operating activities||||
|Depreciation and amortization|**1,488**|1,544|1,474|
|Company pension and postretirement contributions|**(370)**|(967)|(383)|
|Company pension and postretirement expense|**410**|334|250|
|Stock-based compensation expense|**302**|324|298|
|Gain on sale of businesses|**(545)**|(586)|(111)|
|Deferred income taxes|**(57)**|107|7|
|Changes in assets and liabilities||||
|Accounts receivable|**(305)**|(24

In [ ]:
import pymupdf
doc = pymupdf.open(TEST_PDF_PATH)
pix = doc[gold_page_0idx].get_pixmap(dpi=150)
pix.save(f"p^q1wage_{gold_page_0idx}.png")

In [ ]:
# ── Step 2: Chunk each page into 512-token passages ───────────────────────

def count_tokens(text: str) -> int:
    """Count tokens using Stella's own tokenizer (no special tokens added)."""
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_page(
    page_text: str,
    page_num: int,
    doc_name: str,
    max_tokens: int = 512,
    overlap_tokens: int = 50,
) -> list[dict]:
    """
    Split one page's markdown into ≤512-token passages.

    We stay within page boundaries so every chunk has an unambiguous
    page_num — needed to evaluate retrieval against evidence_page_num.

    Algorithm:
      1. Split page text into sentences (NLTK).
      2. Greedily accumulate sentences until token budget would be exceeded.
      3. Flush the chunk; seed the next chunk with the last ~50 tokens
         of the current chunk (the overlap) to smooth the boundary.
    """
    if not page_text.strip():
        return []  # skip blank pages (cover pages, image-only pages, etc.)

    sentences    = sent_tokenize(page_text)
    chunks       = []
    curr_sents   = []
    curr_tokens  = 0

    for sent in sentences:
        sent_toks = count_tokens(sent)

        if curr_tokens + sent_toks > max_tokens and curr_sents:
            # ── Flush current chunk ──
            chunks.append({
                "doc_name"   : doc_name,
                "page_num"   : page_num,
                "chunk_idx"  : len(chunks),
                "text"       : " ".join(curr_sents),
                "token_count": curr_tokens,
            })
            # ── Build overlap seed for the next chunk ──
            # Walk backwards through current sentences, collecting until we
            # accumulate ~overlap_tokens worth of context.
            overlap_sents, overlap_toks = [], 0
            for s in reversed(curr_sents):
                s_toks = count_tokens(s)
                if overlap_toks + s_toks <= overlap_tokens:
                    overlap_sents.insert(0, s)
                    overlap_toks += s_toks
                else:
                    break
            curr_sents  = overlap_sents + [sent]
            curr_tokens = overlap_toks + sent_toks
        else:
            curr_sents.append(sent)
            curr_tokens += sent_toks

    # Flush the final (possibly partial) chunk
    if curr_sents:
        chunks.append({
            "doc_name"   : doc_name,
            "page_num"   : page_num,
            "chunk_idx"  : len(chunks),
            "text"       : " ".join(curr_sents),
            "token_count": curr_tokens,
        })

    return chunks

In [ ]:
# ── Run chunking over all pages of the test document ──────────────────────
print(f"Chunking {len(md_pages)} pages ...")
t0 = time.time()

all_chunks = []
for page_dict in md_pages:
    # page_number from pymupdf4llm is 1-indexed → subtract 1 so our stored
    # page_num is 0-indexed, matching FinanceBench's evidence_page_num exactly.
    pg_num_1idx = page_dict["metadata"]["page_number"]
    pg_num      = pg_num_1idx - 1                    # 0-indexed from here on
    all_chunks.extend(chunk_page(page_dict["text"], pg_num, TEST_DOC_NAME))

chunks_df = pd.DataFrame(all_chunks).reset_index(drop=True)
chunks_df["chunk_id"] = chunks_df.apply(
    lambda r: f"{r.doc_name}__p{r.page_num:04d}_c{r.chunk_idx:02d}", axis=1
)

print(f"Done in {time.time() - t0:.1f}s")
print(f"\nDocument        : {TEST_DOC_NAME}")
print(f"Pages total     : {len(md_pages)}")
print(f"Non-blank pages : {chunks_df.page_num.nunique()}")
print(f"Chunks total    : {len(chunks_df)}")
print(f"Avg tokens/chunk: {chunks_df.token_count.mean():.0f}")
print(f"Max tokens/chunk: {chunks_df.token_count.max()}")

chunks_df[["chunk_id", "page_num", "token_count"]].head(10)

In [ ]:
# ── Critical verification: does the gold evidence page have chunks? ────────
# If this cell shows chunks with meaningful text, Stage 2 is working correctly.
# These are the chunks our retrieval pipeline MUST return for this question.

print(f"Question      : {TEST_QUESTION}")
print(f"Gold answer   : {TEST_ANSWER}")
print(f"Evidence pages: {TEST_EVIDENCE_PAGES}  (0-indexed)")
print()

for gold_page in TEST_EVIDENCE_PAGES:
    hits = chunks_df[chunks_df.page_num == gold_page]
    print(f"=== Page {gold_page} — {len(hits)} chunk(s) ===")
    for _, chunk in hits.iterrows():
        print(f"  chunk_id   : {chunk.chunk_id}")
        print(f"  token_count: {chunk.token_count}")
        print(f"  text preview:")
        print(textwrap.indent(chunk.text[:500], "    "))
        print()

### What you should see above

- The gold evidence page has **at least one chunk** whose preview text visibly contains the answer (or the raw data needed to compute it).
- Token counts are all ≤ 512, most are in the 100–512 range (short pages produce fewer tokens; dense table pages hit the limit and get split).
- The `chunk_id` format `DOCNAME__p0042_c00` encodes everything needed to trace a retrieved chunk back to its exact page — this is how the retrieval evaluator will check hits against `evidence_page_num`.

---

## ✋ End of Stage 2

We now have a `chunks_df` DataFrame with all passages from the test document, each tagged with `doc_name`, `page_num`, `chunk_idx`, `text`, and `token_count`.

**Stage 3** (next session) will take these chunks and:
1. Embed each chunk with **Stella 1.5B** → dense vector index
2. Tokenize each chunk for **BM25** → sparse keyword index

Both indexes together form the retrieval backbone for Stage 5 (hybrid retrieval).

In [ ]:
print("hello world")